# 🎬 Runpod Anime Subber — Jupyter Launcher
شغّل الخلايا بالترتيب داخل **JupyterLab** على Runpod:
- **الخلية 1:** سحب المشروع من GitHub + تثبيت الأدوات والفونتات.
- **الخلية 2:** ربط MEGA و Google Drive في `rclone`.
- **الخلية 3:** تشغيل واجهة Gradio (معاينة + ترجمة + ضغط + رفع MEGA و DoodStream).
- **الخلية 4:** تشغيل **متصفح Chrome داخل Runpod (noVNC على بورت 6080)** عشان تفتح **YourUpload** و **Videa.hu** وترفع الفيديوهات مباشرة من `/workspace/outputs` بسرعة إنترنت Runpod بدون ما تنزل حاجة على جهازك!

In [ ]:
# === الخلية 1: سحب المشروع من GitHub وتثبيت الأدوات والفونتات ===
import os

GITHUB_REPO = "https://github.com/YOUR_USERNAME/runpod-anime-subber.git"
REPO_DIR = "/workspace/runpod-anime-subber"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

%cd {REPO_DIR}
!apt-get update -qq && apt-get install -y -qq aria2 ffmpeg rclone curl fontconfig > /dev/null
!mkdir -p ~/.local/share/fonts && cp -f fonts/*.ttf fonts/*.otf ~/.local/share/fonts/ && fc-cache -fv > /dev/null
!pip install -q -r requirements.txt
print("✅ تم التجهيز بنجاح!")

In [ ]:
# === الخلية 2: ربط MEGA و Google Drive في rclone ===
import os, subprocess

MEGA_EMAIL = ""
MEGA_PASSWORD = ""
GDRIVE_TOKEN = ''''''

sections = []
if MEGA_EMAIL.strip() and MEGA_PASSWORD.strip():
    obs = subprocess.check_output(["rclone", "obscure", MEGA_PASSWORD.strip()], text=True).strip()
    sections.append(f"[mega]\ntype = mega\nuser = {MEGA_EMAIL.strip()}\npass = {obs}\n")
if GDRIVE_TOKEN.strip():
    sections.append(f"[gdrive]\ntype = drive\nscope = drive\ntoken = {GDRIVE_TOKEN.strip()}\n")

if sections:
    os.makedirs(os.path.expanduser("~/.config/rclone"), exist_ok=True)
    with open(os.path.expanduser("~/.config/rclone/rclone.conf"), "w", encoding="utf-8") as f:
        f.write("\n".join(sections))
    print(f"✅ تم حفظ rclone.conf ({len(sections)} remote).")
else:
    print("ℹ️ تم تخطي إعداد rclone حالياً.")

In [ ]:
# === الخلية 3: تشغيل متصفح Chrome داخل Runpod (noVNC على بورت 6080) في الخلفية ===
# تأكد إن بورت 6080 مضاف في Runpod -> Edit Pod -> Expose HTTP Ports (8888,7860,6080)
import os, subprocess, time

VNC_PASS = "123456"  # غيّر كلمة السر لحماية المتصفح

!apt-get update -qq && apt-get install -y -qq xvfb x11vnc fluxbox novnc websockify wget > /dev/null

# تثبيت Google Chrome Stable (يعمل بدون snap داخل Docker)
if subprocess.run(["which", "google-chrome"], capture_output=True).returncode != 0:
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb
    !apt-get install -y -qq /tmp/chrome.deb > /dev/null && rm -f /tmp/chrome.deb

# توجيه الصفحة الرئيسية لـ noVNC مباشرة
!ln -sf /usr/share/novnc/vnc.html /usr/share/novnc/index.html
!mkdir -p /root/.vnc /workspace/chrome_profile /workspace/outputs
!x11vnc -storepasswd {VNC_PASS} /root/.vnc/passwd > /dev/null

# إيقاف أي جلسة قديمة وتشغيل الشاشة الافتراضية + المتصفح + noVNC في الخلفية
!pkill -f "Xvfb :99" || true
!pkill -f "x11vnc" || true
!pkill -f "websockify" || true

os.system("Xvfb :99 -screen 0 1280x800x24 &")
time.sleep(1)
os.system("DISPLAY=:99 fluxbox &")
os.system("x11vnc -display :99 -rfbauth /root/.vnc/passwd -listen 127.0.0.1 -rfbport 5900 -forever -shared &")
os.system("websockify --web=/usr/share/novnc 0.0.0.0:6080 127.0.0.1:5900 &")
time.sleep(1)
os.system(
    "DISPLAY=:99 google-chrome --no-sandbox --disable-dev-shm-usage "
    "--user-data-dir=/workspace/chrome_profile "
    "https://www.yourupload.com https://videa.hu &"
)

print("✅ المتصفح شغّال في الخلفية! افتحه من Runpod -> Connect -> HTTP Service [Port 6080]")
print(f"🔑 كلمة سر VNC: {VNC_PASS}")
print("📁 عند الرفع من YourUpload أو Videa.hu، اختار الفيديوهات من المسار: /workspace/outputs")

In [ ]:
# === الخلية 4: تشغيل واجهة Gradio (بورت 7860) ===
import os

os.environ["GEMINI_API_KEY"] = ""
os.environ["DOODSTREAM_API_KEY"] = ""
os.environ["UI_USER"] = "admin"
os.environ["UI_PASS"] = "123456"
os.environ["GRADIO_SHARE"] = "true"
os.environ["PORT"] = "7860"
os.environ["RUNPOD_OUTPUT_ROOT"] = "/workspace/outputs"

!python3 /workspace/runpod-anime-subber/app.py